In [ ]:
import pandas as pd

file_path = "/content/Covid Data.csv"
df = pd.read_csv(file_path)
print(df.head())

ModuleNotFoundError: No module named 'pandas'

: 

### Data Cleaning and Preprocessing

First, let's get an overview of the dataset's structure, including data types and non-null values, to identify areas that need cleaning.

In [ ]:
df.info()

From `df.info()`, we can see that most columns are integers. However, `DATE_DIED` is an object type, and based on `df.head()`, it contains date strings and potentially other values like '97'.

Let's examine the unique values in `DATE_DIED` and other relevant columns that might contain '97' or '99' as special codes (often representing 'not applicable' or 'unknown').

In [ ]:
print('Unique values for DATE_DIED:', df['DATE_DIED'].unique())

# Identify other columns that might contain '97' or '99' special codes
columns_to_check = ['INTUBED', 'PREGNANT', 'ICU', 'PNEUMONIA', 'DIABETES', 'COPD', 'ASTHMA', 'INMSUPR', 'HIPERTENSION', 'OTHER_DISEASE', 'CARDIOVASCULAR', 'OBESITY', 'RENAL_CHRONIC', 'TOBACCO']
for col in columns_to_check:
    if 97 in df[col].unique() or 99 in df[col].unique():
        print(f'Unique values for {col}: {df[col].unique()}')

The unique values reveal that '9999-99-99' in `DATE_DIED` and '97', '99' in other columns are likely codes for 'not applicable' or 'unknown'. We should treat these as missing values (NaN) for proper analysis.

Let's replace these codes with `NaN` and then convert `DATE_DIED` to a proper datetime format. For binary columns (like `PNEUMONIA`, `DIABETES`, etc.), '1' typically means 'yes' and '2' means 'no'. We'll map these to 1 and 0 for consistency, and handle '97'/'99' as NaNs.

In [ ]:
# Replace '9999-99-99' in DATE_DIED with NaN
df['DATE_DIED'] = df['DATE_DIED'].replace('9999-99-99', pd.NA)

# Convert DATE_DIED to datetime format
df['DATE_DIED'] = pd.to_datetime(df['DATE_DIED'], errors='coerce')

# Define columns with '97' and '99' as special codes (representing 'No/Missing')
# 'SEX' has 1 (female) and 2 (male). 'PATIENT_TYPE' has 1 (outpatient) and 2 (inpatient).
# 'USMER' is 1 (yes) and 2 (no)
binary_cols_with_special_codes = [
    'INTUBED', 'PNEUMONIA', 'PREGNANT', 'DIABETES', 'COPD', 'ASTHMA',
    'INMSUPR', 'HIPERTENSION', 'OTHER_DISEASE', 'CARDIOVASCULAR', 'OBESITY',
    'RENAL_CHRONIC', 'TOBACCO', 'ICU'
]

# For these columns, '1' means 'Yes', '2' means 'No'. '97'/'99' means 'Not applicable/Unknown'.
# Let's map '1' to 1 (Yes), '2' to 0 (No), and '97'/'99' to NaN.
for col in binary_cols_with_special_codes:
    df[col] = df[col].replace({1: 1, 2: 0, 97: pd.NA, 99: pd.NA})

# For 'SEX': 1 = Female, 2 = Male. Assuming this is consistent with domain knowledge.
# Let's keep 1 and 2, but we might convert to categorical or one-hot later.
# For 'USMER': 1 = Yes, 2 = No. Convert to 1 (Yes) and 0 (No).
df['USMER'] = df['USMER'].replace({1: 1, 2: 0})

# For 'PATIENT_TYPE': 1 = Outpatient, 2 = Inpatient. Convert to 1 (Outpatient) and 0 (Inpatient).
df['PATIENT_TYPE'] = df['PATIENT_TYPE'].replace({1: 1, 2: 0})

# Recheck info to see updated dtypes and non-nulls
df.info()

Now that we've handled the special codes and converted `DATE_DIED` to datetime, let's check the number of missing values (NaNs) across all columns to understand the extent of data loss or unknown values.

In [ ]:
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
})

display(missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False))

We can observe a significant percentage of missing values in `INTUBED`, `ICU`, and `DATE_DIED`. `PREGNANT` also has a high percentage, likely because it's only applicable to females.

For `INTUBED` and `ICU`, dropping these rows might lead to substantial data loss given they have over 80% missing values. It's often better to consider these as separate categories if the missingness itself holds information (e.g., 'not intubated' or 'not in ICU' if `97` meant 'not applicable'). However, since we've converted them to `NaN`, we need to decide on imputation or dropping.

For `PREGNANT`, the high missing rate is expected. It makes sense to impute missing values for 'PREGNANT' to '0' (not pregnant) for males, or keep it as `NaN` and handle it during modeling. Let's start by addressing `PREGNANT` by imputing `NaN` with 0, assuming that `NaN` predominantly indicates 'not pregnant' in cases where it's not explicitly recorded, especially for males (which we will verify after filling).

Then, for `INTUBED` and `ICU`, given the high percentage of NaNs, we can consider filling NaNs with 0 (meaning 'not intubated' or 'not in ICU') as a first approach, assuming that '97'/'99' (which we replaced with `NaN`) implied absence of these conditions.

Let's apply these imputations and then create a target variable for COVID-19 related death.

In [ ]:
# Impute 'PREGNANT' NaNs with 0 (not pregnant)
df['PREGNANT'] = df['PREGNANT'].fillna(0)

# Impute 'INTUBED' and 'ICU' NaNs with 0 (not intubated, not in ICU)
df['INTUBED'] = df['INTUBED'].fillna(0)
df['ICU'] = df['ICU'].fillna(0)

# Create a target variable: 'DECEASED' (1 if died, 0 if not died)
# If DATE_DIED is not NA, it means the patient died. Otherwise, they did not.
df['DECEASED'] = df['DATE_DIED'].apply(lambda x: 0 if pd.isna(x) else 1)

# Drop the original DATE_DIED column as we have our target variable now
df = df.drop(columns=['DATE_DIED'])

# Check the number of missing values again after imputation and creating target variable
missing_values_after = df.isnull().sum()
missing_percentage_after = (df.isnull().sum() / len(df)) * 100

missing_df_after = pd.DataFrame({
    'Missing Count': missing_values_after,
    'Missing Percentage': missing_percentage_after
})

display(missing_df_after[missing_df_after['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False))

df.head()

It appears `PNEUMONIA` still has some missing values. Given that '1' means 'Yes' and '0' means 'No' for this binary feature, it's reasonable to impute the remaining `NaN`s with `0`, assuming that the absence of a record implies no pneumonia.

In [ ]:
# Impute 'PNEUMONIA' NaNs with 0 (no pneumonia)
df['PNEUMONIA'] = df['PNEUMONIA'].fillna(0)

# Ensure all binary columns are of integer type now
# PNEUMONIA was object type because of the NaNs, converting it to int/float as appropriate
df['PNEUMONIA'] = df['PNEUMONIA'].astype(int)

# Check for any remaining missing values
missing_values_final = df.isnull().sum()
missing_percentage_final = (df.isnull().sum() / len(df)) * 100

missing_df_final = pd.DataFrame({
    'Missing Count': missing_values_final,
    'Missing Percentage': missing_percentage_final
})

display(missing_df_final[missing_df_final['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False))

df.info()

Now that we've addressed all missing values, let's consider other preprocessing steps:

*   **Categorical Features:** `SEX`, `CLASIFFICATION_FINAL`, and `MEDICAL_UNIT` are currently numerical but represent categories. We might want to one-hot encode them if we're using models that require numerical inputs and don't handle categorical features natively.
*   **Numerical Features:** `AGE` is a numerical feature. Depending on the model, it might benefit from scaling (e.g., standardization or normalization).
*   **Correlation Analysis:** We could explore correlations between features and the target variable (`DECEASED`) to identify important predictors.

Let's start by looking at the unique values and distributions of `SEX`, `CLASIFFICATION_FINAL`, and `MEDICAL_UNIT` to decide on the best encoding strategy.

In [ ]:
print('Unique values for SEX:', df['SEX'].unique())
print('Value counts for SEX:\n', df['SEX'].value_counts())
print('\nUnique values for CLASIFFICATION_FINAL:', df['CLASIFFICATION_FINAL'].unique())
print('Value counts for CLASIFFICATION_FINAL:\n', df['CLASIFFICATION_FINAL'].value_counts())
print('\nUnique values for MEDICAL_UNIT:', df['MEDICAL_UNIT'].unique())
print('Value counts for MEDICAL_UNIT:\n', df['MEDICAL_UNIT'].value_counts())

It appears `PNEUMONIA` still has some missing values. Given that '1' means 'Yes' and '0' means 'No' for this binary feature, it's reasonable to impute the remaining `NaN`s with `0`, assuming that the absence of a record implies no pneumonia.

In [ ]:
# Impute 'PNEUMONIA' NaNs with 0 (no pneumonia)
df['PNEUMONIA'] = df['PNEUMONIA'].fillna(0)

# Ensure all binary columns are of integer type now
# PNEUMONIA was object type because of the NaNs, converting it to int/float as appropriate
df['PNEUMONIA'] = df['PNEUMONIA'].astype(int)

# Check for any remaining missing values
missing_values_final = df.isnull().sum()
missing_percentage_final = (df.isnull().sum() / len(df)) * 100

missing_df_final = pd.DataFrame({
    'Missing Count': missing_values_final,
    'Missing Percentage': missing_percentage_final
})

display(missing_df_final[missing_df_final['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False))

df.info()

Now that we've addressed all missing values, let's consider other preprocessing steps:

*   **Categorical Features:** `SEX`, `CLASIFFICATION_FINAL`, and `MEDICAL_UNIT` are currently numerical but represent categories. We might want to one-hot encode them if we're using models that require numerical inputs and don't handle categorical features natively.
*   **Numerical Features:** `AGE` is a numerical feature. Depending on the model, it might benefit from scaling (e.g., standardization or normalization).
*   **Correlation Analysis:** We could explore correlations between features and the target variable (`DECEASED`) to identify important predictors.

Let's start by looking at the unique values and distributions of `SEX`, `CLASIFFICATION_FINAL`, and `MEDICAL_UNIT` to decide on the best encoding strategy.

In [ ]:
print('Unique values for SEX:', df['SEX'].unique())
print('Value counts for SEX:\n', df['SEX'].value_counts())
print('\nUnique values for CLASIFFICATION_FINAL:', df['CLASIFFICATION_FINAL'].unique())
print('Value counts for CLASIFFICATION_FINAL:\n', df['CLASIFFICATION_FINAL'].value_counts())
print('\nUnique values for MEDICAL_UNIT:', df['MEDICAL_UNIT'].unique())
print('Value counts for MEDICAL_UNIT:\n', df['MEDICAL_UNIT'].value_counts())

It appears `PNEUMONIA` still has some missing values. Given that '1' means 'Yes' and '0' means 'No' for this binary feature, it's reasonable to impute the remaining `NaN`s with `0`, assuming that the absence of a record implies no pneumonia.

In [ ]:
# Impute 'PNEUMONIA' NaNs with 0 (no pneumonia)
df['PNEUMONIA'] = df['PNEUMONIA'].fillna(0)

# Ensure all binary columns are of integer type now
# PNEUMONIA was object type because of the NaNs, converting it to int/float as appropriate
df['PNEUMONIA'] = df['PNEUMONIA'].astype(int)

# Check for any remaining missing values
missing_values_final = df.isnull().sum()
missing_percentage_final = (df.isnull().sum() / len(df)) * 100

missing_df_final = pd.DataFrame({
    'Missing Count': missing_values_final,
    'Missing Percentage': missing_percentage_final
})

display(missing_df_final[missing_df_final['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False))

df.info()

Now that we've addressed all missing values, let's consider other preprocessing steps:

*   **Categorical Features:** `SEX`, `CLASIFFICATION_FINAL`, and `MEDICAL_UNIT` are currently numerical but represent categories. We might want to one-hot encode them if we're using models that require numerical inputs and don't handle categorical features natively.
*   **Numerical Features:** `AGE` is a numerical feature. Depending on the model, it might benefit from scaling (e.g., standardization or normalization).
*   **Correlation Analysis:** We could explore correlations between features and the target variable (`DECEASED`) to identify important predictors.

Let's start by looking at the unique values and distributions of `SEX`, `CLASIFFICATION_FINAL`, and `MEDICAL_UNIT` to decide on the best encoding strategy.

In [ ]:
print('Unique values for SEX:', df['SEX'].unique())
print('Value counts for SEX:\n', df['SEX'].value_counts())
print('\nUnique values for CLASIFFICATION_FINAL:', df['CLASIFFICATION_FINAL'].unique())
print('Value counts for CLASIFFICATION_FINAL:\n', df['CLASIFFICATION_FINAL'].value_counts())
print('\nUnique values for MEDICAL_UNIT:', df['MEDICAL_UNIT'].unique())
print('Value counts for MEDICAL_UNIT:\n', df['MEDICAL_UNIT'].value_counts())

All missing values have been addressed. Now, we will one-hot encode the categorical features: `SEX`, `CLASIFFICATION_FINAL`, and `MEDICAL_UNIT`.

In [ ]:
# One-hot encode 'SEX', 'CLASIFFICATION_FINAL', and 'MEDICAL_UNIT'
df = pd.get_dummies(df, columns=['SEX', 'CLASIFFICATION_FINAL', 'MEDICAL_UNIT'], drop_first=True)

# Display the first few rows and the new shape of the DataFrame
print(df.head())
print(f"\nNew DataFrame shape: {df.shape}")

As confirmed by the previous steps, all missing values have been addressed, and categorical features have been one-hot encoded. Let's look at the head and info of the DataFrame to see its final cleaned state.

In [ ]:
print(df.head())
df.info()

The dataset is now thoroughly cleaned and preprocessed. All missing values have been handled, and categorical features have been converted using one-hot encoding. Next, we could consider scaling numerical features like `AGE` if your chosen model requires it, or proceed with correlation analysis and model building.

### Exploratory Data Analysis (EDA)

Let's start by examining the distribution of our target variable, `DECEASED`, to understand the class balance.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the target variable 'DECEASED'
plt.figure(figsize=(6, 4))
sns.countplot(x='DECEASED', data=df, palette='viridis')
plt.title('Distribution of Deceased Status')
plt.xlabel('Deceased (0: No, 1: Yes)')
plt.ylabel('Count')
plt.show()

print("Deceased status value counts:\n", df['DECEASED'].value_counts())
print("\nDeceased status percentage:\n", df['DECEASED'].value_counts(normalize=True) * 100)

The target variable `DECEASED` shows a significant class imbalance, with a much smaller number of deceased cases (around 2.9%). This is important to note for model building, as we might need to use techniques to handle imbalanced datasets.

Next, let's look at the distribution of `AGE` and see how it relates to the `DECEASED` status.

In [ ]:
# Distribution of Age
plt.figure(figsize=(10, 6))
sns.histplot(df['AGE'], bins=30, kde=True, color='skyblue')
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

# Age distribution by Deceased Status
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='AGE', hue='DECEASED', bins=30, kde=True, palette='coolwarm')
plt.title('Age Distribution by Deceased Status')
plt.xlabel('Age')
plt.ylabel('Count')
plt.legend(title='Deceased', labels=['No', 'Yes'])
plt.show()

The age distribution shows a typical bell curve. When segmented by deceased status, it appears that older individuals have a higher likelihood of being deceased, which aligns with general understanding of COVID-19 impact.

Now, let's examine the distributions of `SEX` (represented by `SEX_2` for males) and other key health conditions in relation to `DECEASED` status.

In [ ]:
# Distribution of Sex and Deceased Status (SEX_2: True for Male, False for Female)
plt.figure(figsize=(8, 5))
sns.countplot(x='SEX_2', hue='DECEASED', data=df, palette='pastel')
plt.title('Deceased Status by Sex (SEX_2: 0=Female, 1=Male)')
plt.xlabel('Sex (0=Female, 1=Male)')
plt.ylabel('Count')
plt.legend(title='Deceased', labels=['No', 'Yes'])
plt.xticks(ticks=[0, 1], labels=['Female', 'Male'])
plt.show()

# Distribution of PNEUMONIA and Deceased Status
plt.figure(figsize=(8, 5))
sns.countplot(x='PNEUMONIA', hue='DECEASED', data=df, palette='pastel')
plt.title('Deceased Status by Pneumonia (0=No, 1=Yes)')
plt.xlabel('Pneumonia (0=No, 1=Yes)')
plt.ylabel('Count')
plt.legend(title='Deceased', labels=['No', 'Yes'])
plt.show()

# Distribution of DIABETES and Deceased Status
plt.figure(figsize=(8, 5))
sns.countplot(x='DIABETES', hue='DECEASED', data=df, palette='pastel')
plt.title('Deceased Status by Diabetes (0=No, 1=Yes)')
plt.xlabel('Diabetes (0=No, 1=Yes)')
plt.ylabel('Count')
plt.legend(title='Deceased', labels=['No', 'Yes'])
plt.show()

The visualizations above give us insights into how sex, pneumonia, and diabetes relate to the deceased status. We can observe that individuals with pneumonia or diabetes tend to have a higher count of deceased cases.

Finally, let's look at the correlation matrix of our features, especially with the `DECEASED` target variable. This will help us identify features that are strongly positively or negatively correlated with the outcome.

In [ ]:
# Calculate the correlation matrix
correlation_matrix = df.corr(numeric_only=True)

# Plot the heatmap
plt.figure(figsize=(18, 15))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Features')
plt.show()

The correlation heatmap provides a comprehensive view of the relationships between all features. From this, we can identify features that have a strong correlation with `DECEASED`, as well as multicollinearity among predictor variables. For instance, `PATIENT_TYPE` (inpatient) and `INTUBED` seem to have a relatively strong positive correlation with `DECEASED`. `AGE` also shows a positive correlation.

This completes our initial EDA. Would you like to explore any specific relationships further, or perhaps move on to feature engineering or model building?

### Feature Engineering: Scaling Numerical Features

Numerical features, especially those with a wide range of values like `AGE`, can sometimes disproportionately influence models that rely on distance metrics (e.g., K-Nearest Neighbors, Support Vector Machines) or gradient descent (e.g., neural networks). Scaling these features ensures that they all contribute equally to the model's performance.

We will use `StandardScaler` to transform the `AGE` column, which will normalize its distribution to have a mean of 0 and a standard deviation of 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize StandardScaler
scaler = StandardScaler()

# Reshape 'AGE' column for scaling (StandardScaler expects 2D array)
df['AGE_scaled'] = scaler.fit_transform(df[['AGE']])

# Display the first few rows with the new 'AGE_scaled' column and descriptive statistics
print(df[['AGE', 'AGE_scaled']].head())
print(f"\nDescriptive statistics for scaled AGE:\n{df['AGE_scaled'].describe()}")

### Feature Engineering: Scaling Numerical Features

Numerical features, especially those with a wide range of values like `AGE`, can sometimes disproportionately influence models that rely on distance metrics (e.g., K-Nearest Neighbors, Support Vector Machines) or gradient descent (e.g., neural networks). Scaling these features ensures that they all contribute equally to the model's performance.

We will use `StandardScaler` to transform the `AGE` column, which will normalize its distribution to have a mean of 0 and a standard deviation of 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize StandardScaler
scaler = StandardScaler()

# Reshape 'AGE' column for scaling (StandardScaler expects 2D array)
df['AGE_scaled'] = scaler.fit_transform(df[['AGE']])

# Display the first few rows with the new 'AGE_scaled' column and descriptive statistics
print(df[['AGE', 'AGE_scaled']].head())
print(f"\nDescriptive statistics for scaled AGE:\n{df['AGE_scaled'].describe()}")

### Data Splitting: Training and Testing Sets

To prepare for model building, we need to split our dataset into features (X) and the target variable (y). We will then further divide these into training and testing sets. This ensures that we can evaluate our model's performance on data it has not seen during training, providing a more reliable measure of its generalization ability.

We will use `DECEASED` as our target variable and the remaining columns as features. A common split ratio is 80% for training and 20% for testing.

In [ ]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df.drop(columns=['DECEASED', 'AGE']) # Drop original AGE as we now have AGE_scaled
y = df['DECEASED']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Original dataset shape: {df.shape}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

print("\nDistribution of DECEASED in training set:")
print(y_train.value_counts(normalize=True) * 100)

print("\nDistribution of DECEASED in test set:")
print(y_test.value_counts(normalize=True) * 100)

### Training Features (`X_train`)

Below are the feature columns used for model training. These include the binary health indicators and the one-hot encoded categorical variables.

In [ ]:
# List all feature columns in X_train
print("Features used for training:")
print(X_train.columns.tolist())

# Display the first 5 rows of the training features
display(X_train.head())

### Model Building: Logistic Regression

Given the data's preparation, including handling class imbalance, we can now proceed to build our first predictive model. Logistic Regression is a suitable choice for binary classification tasks, and it's a good baseline to establish performance before exploring more complex models.

We will train a Logistic Regression model on our `X_train` and `y_train` data. To address the class imbalance, we'll use the `class_weight='balanced'` parameter, which automatically adjusts weights inversely proportional to class frequencies in the input data. This helps the model pay more attention to the minority class.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Initialize the Logistic Regression model with class_weight='balanced'
# and a higher max_iter for convergence, and a random_state for reproducibility
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# Train the model on the training data
model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

### Model Evaluation: Logistic Regression

Now that our model is trained, we need to evaluate its performance on the unseen test data. We will use several metrics relevant for binary classification, particularly considering the class imbalance in our target variable `DECEASED`.

Key metrics include:
- **Accuracy**: Overall correctness of the model.
- **Precision**: The proportion of positive identifications that were actually correct.
- **Recall (Sensitivity)**: The proportion of actual positives that were identified correctly.
- **F1-Score**: The harmonic mean of precision and recall, useful for imbalanced classes.
- **ROC AUC Score**: Measures the area under the Receiver Operating Characteristic curve, indicating the model's ability to distinguish between classes.
- **Confusion Matrix**: Provides a detailed breakdown of correct and incorrect classifications.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Make predictions on the test set
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1] # Probability of the positive class

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"ROC AUC Score: {roc_auc:.4f}")

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted No', 'Predicted Yes'],
            yticklabels=['Actual No', 'Actual Yes'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
import pickle

# Save the Logistic Regression model to a file
model_filename = 'covid19Model.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(model, file)

print(f'Model saved successfully as {model_filename}')

### Model Building: Random Forest Classifier

To improve upon the baseline Logistic Regression model and potentially achieve higher accuracy, we will now implement a Random Forest Classifier. Random Forests are ensemble learning methods known for their robustness and ability to handle complex relationships within data. They are particularly effective for imbalanced datasets when `class_weight='balanced'` is utilized.

We will train the Random Forest model on the `X_train` and `y_train` datasets and then evaluate its performance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the Random Forest Classifier model with class_weight='balanced'
# and a random_state for reproducibility
# n_estimators can be tuned, starting with a reasonable number like 100
model_rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)

# Train the model on the training data
model_rf.fit(X_train, y_train)

print("Random Forest Classifier model trained successfully!")

### Model Evaluation: Random Forest Classifier

After training the Random Forest model, we will evaluate its performance on the test set. We will use the same set of metrics (accuracy, precision, recall, F1-score, ROC AUC, and confusion matrix) to compare its effectiveness with the Logistic Regression model.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Make predictions on the test set using the Random Forest model
y_pred_rf = model_rf.predict(X_test)
y_pred_proba_rf = model_rf.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics for Random Forest
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)

print(f"Random Forest Accuracy: {accuracy_rf:.4f}")
print(f"Random Forest Precision: {precision_rf:.4f}")
print(f"Random Forest Recall: {recall_rf:.4f}")
print(f"Random Forest F1-Score: {f1_rf:.4f}")
print(f"Random Forest ROC AUC Score: {roc_auc_rf:.4f}")

# Plot Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Predicted No', 'Predicted Yes'],
            yticklabels=['Actual No', 'Actual Yes'])
plt.title('Random Forest Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report

# Calculate scale_pos_weight for imbalanced data
# count(negative) / count(positive)
scale_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

# Initialize XGBoost Classifier
model_xgb = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Train the model
model_xgb.fit(X_train, y_train)

print("XGBoost model trained successfully!")

In [ ]:
# Make predictions
y_pred_xgb = model_xgb.predict(X_test)
y_pred_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb)
recall_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)
roc_auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)

print(f"XGBoost Accuracy: {accuracy_xgb:.4f}")
print(f"XGBoost Precision: {precision_xgb:.4f}")
print(f"XGBoost Recall: {recall_xgb:.4f}")
print(f"XGBoost F1-Score: {f1_xgb:.4f}")
print(f"XGBoost ROC AUC Score: {roc_auc_xgb:.4f}")

# Plot Confusion Matrix
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges', cbar=False,
            xticklabels=['Predicted No', 'Predicted Yes'],
            yticklabels=['Actual No', 'Actual Yes'])
plt.title('XGBoost Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

### Model Building: Gradient Boosting

Gradient Boosting is another powerful ensemble method. Since we have a large dataset, we will use `HistGradientBoostingClassifier`, which is optimized for speed and works well with imbalanced data using the `class_weight='balanced'` equivalent logic through sample weights or by handling it during the boosting process.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

# Initialize and train the HistGradientBoostingClassifier
# It handles class imbalance implicitly well, but we can use sample_weights if needed.
# For now, we'll use standard parameters as a strong baseline.
model_gb = HistGradientBoostingClassifier(random_state=42)

model_gb.fit(X_train, y_train)

print("Gradient Boosting model trained successfully!")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Evaluation
y_pred_gb = model_gb.predict(X_test)
y_pred_proba_gb = model_gb.predict_proba(X_test)[:, 1]

accuracy_gb = accuracy_score(y_test, y_pred_gb)
precision_gb = precision_score(y_test, y_pred_gb)
recall_gb = recall_score(y_test, y_pred_gb)
f1_gb = f1_score(y_test, y_pred_gb)
roc_auc_gb = roc_auc_score(y_test, y_pred_proba_gb)

print(f"Gradient Boosting Accuracy: {accuracy_gb:.4f}")
print(f"Gradient Boosting Precision: {precision_gb:.4f}")
print(f"Gradient Boosting Recall: {recall_gb:.4f}")
print(f"Gradient Boosting F1-Score: {f1_gb:.4f}")
print(f"Gradient Boosting ROC AUC Score: {roc_auc_gb:.4f}")

# Plot Confusion Matrix
cm_gb = confusion_matrix(y_test, y_pred_gb)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_gb, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=['Predicted No', 'Predicted Yes'],
            yticklabels=['Actual No', 'Actual Yes'])
plt.title('Gradient Boosting Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

We have now completed the initial data cleaning and preprocessing steps, including handling missing values, converting data types, and encoding categorical features. The dataset is ready for further analysis or model building.